# MODUL OCR - SISTEM VERIFIKASI DOKUMEN OTOMATIS

**Versi Windows 11**

---

## Deskripsi
Ekstraksi teks dari dokumen PDF/Gambar menggunakan Tesseract OCR

**Alur Kerja:**
```
Input Gambar → Orientation/Skew Correction → Grayscale → Adaptive Denoise → Tesseract OCR (Grayscale, PSM 3, OEM 3) → Post-processing (garbage line removal) → Single-line Evaluation
```

**Catatan Pipeline:**
- CLAHE, Sharpening, Otsu Thresholding, dan Morphology **dinonaktifkan** (pass-through)
- Tesseract LSTM (OEM 3) bekerja lebih akurat pada gambar **grayscale** dibanding binary
- **Post-processing**: Garbage line removal untuk membersihkan artefak watermark/stempel
- Evaluasi menggunakan mode **single-line** (layout/newline diabaikan)

**Target Performa:** <10 detik per dokumen

**Framework:** Python + Tesseract OCR + OpenCV

OCR System dengan Pengujian Akurasi, WER, dan CER

Ditambahkan: Testing menggunakan jiwer (alternatif FastWER yang lebih mudah di Windows)

In [1]:
# =============================================
# INSTALASI PYTHON PACKAGES
# =============================================

!pip install pytesseract pdf2image Pillow opencv-python python-Levenshtein jiwer

print("\n✅ Instalasi package selesai!")
print("⚠️  Pastikan Tesseract OCR dan Poppler sudah terinstall!")

Defaulting to user installation because normal site-packages is not writeable

✅ Instalasi package selesai!
⚠️  Pastikan Tesseract OCR dan Poppler sudah terinstall!


In [2]:
# ================================================
# KONFIGURASI PATH TESSERACT & POPPLER
# ================================================
import os

# SESUAIKAN PATH INI DENGAN LOKASI INSTALASI ANDA!
TESSERACT_PATH = r'C:\Program Files\Tesseract-OCR\tesseract.exe'
POPPLER_PATH = r'C:\Program Files\poppler-25.07.0\Library\bin'

# Verifikasi path Tesseract
if os.path.exists(TESSERACT_PATH):
    print(f"✅ Tesseract ditemukan di: {TESSERACT_PATH}")
else:
    print(f"❌ Tesseract TIDAK ditemukan di: {TESSERACT_PATH}")
    print("⚠️  Sesuaikan TESSERACT_PATH dengan lokasi instalasi Anda!")

# Verifikasi path Poppler
if os.path.exists(POPPLER_PATH):
    print(f"✅ Poppler ditemukan di: {POPPLER_PATH}")
else:
    print(f"❌ Poppler TIDAK ditemukan di: {POPPLER_PATH}")
    print("⚠️  Sesuaikan POPPLER_PATH dengan lokasi instalasi Anda!")

✅ Tesseract ditemukan di: C:\Program Files\Tesseract-OCR\tesseract.exe
✅ Poppler ditemukan di: C:\Program Files\poppler-25.07.0\Library\bin


In [3]:
# ================================================
# IMPORT LIBRARY DENGAN ERROR HANDLING
# ================================================
try:
    import pytesseract
    from pdf2image import convert_from_path
    from IPython.display import display
    from PIL import Image
    import io
    import os
    import cv2
    import numpy as np
    import Levenshtein
    import re
    import json
    import time
    from pathlib import Path
    from jiwer import wer, cer  # Import jiwer untuk WER dan CER

    # Set Tesseract path untuk Windows
    pytesseract.pytesseract.tesseract_cmd = TESSERACT_PATH

    print("✅ Semua library berhasil di-import")
    print("📦 Menggunakan jiwer untuk WER dan CER calculation")

except ImportError as e:
    print(f"❌ Error importing library: {e}")
    print("⚠️  Pastikan semua library sudah terinstall")
    raise

✅ Semua library berhasil di-import
📦 Menggunakan jiwer untuk WER dan CER calculation


In [4]:
# ================================================
# VERIFIKASI TESSERACT OCR ENGINE
# ================================================
try:
    tesseract_version = pytesseract.get_tesseract_version()
    print(f"✅ Tesseract engine version: {tesseract_version}")
    
    # Cek bahasa yang tersedia
    languages = pytesseract.get_languages()
    print(f"\n📚 Bahasa yang tersedia: {', '.join(languages)}")
    
    if 'ind' in languages:
        print("✅ Bahasa Indonesia tersedia")
    else:
        print("⚠️  Bahasa Indonesia tidak tersedia")
        print("   Download tessdata dari: https://github.com/tesseract-ocr/tessdata")
        
except pytesseract.TesseractNotFoundError:
    print("❌ Tesseract tidak ditemukan!")
    print(f"   Path yang dicoba: {TESSERACT_PATH}")
    print("   Pastikan Tesseract sudah terinstall dan path sudah benar")
    raise

✅ Tesseract engine version: 5.5.0.20241111

📚 Bahasa yang tersedia: eng, ind, osd
✅ Bahasa Indonesia tersedia


In [ ]:
# ================================================
# KONFIGURASI GROUND TRUTH DARI FILE TXT LOKAL
# ================================================

import os
from pathlib import Path

# SESUAIKAN PATH INI KE FOLDER GROUND TRUTH ANDA
# GROUND_TRUTH_FOLDER = r'E:\Softwares\Jupyter\Projects\OCR\dokumen\dokumen_normal\ground_truth'
BASE_FOLDER = r'E:\Softwares\Jupyter\Projects\OCR\data\dokumen'

# User tinggal pilih folder kategori
CATEGORY = 'listrik'  # atau 'dokumen_blur', 'dokumen_noisy', dll
GROUND_TRUTH_FOLDER = os.path.join(BASE_FOLDER, CATEGORY, 'ground_truth')
DOCUMENTS_FOLDER = os.path.join(BASE_FOLDER, CATEGORY)

# Batasi jumlah ground truth yang ditampilkan (None = tampilkan semua)
MAX_DISPLAY_GT = 5

print("🔍 Membaca ground truth dari file lokal...")
print("=" * 60)

GROUND_TRUTH = {}

# Cek apakah folder exists
if not os.path.exists(GROUND_TRUTH_FOLDER):
    print(f"❌ Folder tidak ditemukan: {GROUND_TRUTH_FOLDER}")
    print("⚠️  Sesuaikan GROUND_TRUTH_FOLDER dengan lokasi folder Anda")
else:
    print(f"✅ Folder ditemukan: {GROUND_TRUTH_FOLDER}\n")
    
    # Baca semua file .txt di folder
    txt_files = list(Path(GROUND_TRUTH_FOLDER).glob('*.txt'))
    
    if not txt_files:
        print(f"⚠️  Tidak ada file .txt ditemukan di folder")
    else:
        for i_gt, txt_file in enumerate(txt_files):
            try:
                # Baca isi file
                with open(txt_file, 'r', encoding='utf-8') as f:
                    content = f.read()
                
                # Dapatkan nama file tanpa path
                filename_base = txt_file.stem  # Nama file tanpa extension
                
                # Cari file dokumen yang sesuai di folder yang sama
                import glob
                doc_folder = os.path.dirname(GROUND_TRUTH_FOLDER)
                
                # Coba cocokkan dengan PDF atau gambar
                possible_extensions = ['.pdf', '.jpg', '.jpeg', '.png']
                actual_file = None
                
                for ext in possible_extensions:
                    pattern = os.path.join(doc_folder, f"{filename_base}{ext}")
                    matches = glob.glob(pattern)
                    if matches:
                        actual_file = os.path.basename(matches[0])
                        break
                
                # Jika tidak ketemu file asli, gunakan PDF sebagai default
                if actual_file is None:
                    actual_file = f"{filename_base}.pdf"
                
                key = actual_file
                GROUND_TRUTH[key] = content
                
                # Tampilkan log loading (dibatasi MAX_DISPLAY_GT)
                char_count = len(content)
                if MAX_DISPLAY_GT is None or i_gt < MAX_DISPLAY_GT:
                    print(f"✅ {txt_file.name} ({char_count} karakter)")
                    print(f"   → Key: {key}")
                elif i_gt == MAX_DISPLAY_GT:
                    print(f"   ... dan {len(txt_files) - MAX_DISPLAY_GT} file lainnya")
                
            except Exception as e:
                print(f"❌ Error membaca {txt_file.name}: {e}")

print("\n" + "=" * 60)
print(f"📊 Total ground truth dimuat: {len(GROUND_TRUTH)} file")
print("=" * 60)

if GROUND_TRUTH:
    print("\n📋 Daftar ground truth yang tersedia:")
    gt_items = list(GROUND_TRUTH.items())
    for i_gt, (filename, content) in enumerate(gt_items):
        if MAX_DISPLAY_GT is not None and i_gt >= MAX_DISPLAY_GT:
            print(f"   ... dan {len(gt_items) - MAX_DISPLAY_GT} lainnya (tidak ditampilkan)")
            break
        char_count = len(content)
        line_count = content.count('\n') + 1
        print(f"   • {filename}: {char_count} karakter, {line_count} baris")
    
    print("\n⚠️  PENTING:")
    print("   1. Pastikan nama file .txt sesuai dengan nama dokumen yang akan di-OCR")
    print("   2. Contoh: 'Struk 1.txt' untuk 'Struk 1.pdf'")
    print("   3. Ground truth harus berisi teks RAW tanpa normalisasi")
else:
    print("\n⚠️  Tidak ada ground truth yang dimuat!")
    print("   Silakan periksa:")
    print(f"   1. Path folder: {GROUND_TRUTH_FOLDER}")
    print("   2. Pastikan ada file .txt di folder tersebut")

In [ ]:
# ================================================
# INPUT FILE
# ================================================
# ✅ SOLUSI (Otomatis - scan folder)
from pathlib import Path

# Batasi jumlah input file yang ditampilkan detail-nya (None = tampilkan semua)
MAX_DISPLAY_INPUT = 5

def get_all_documents(folder_path, extensions=['.pdf', '.jpg', '.jpeg', '.png']):
    """Otomatis ambil semua file dokumen dari folder"""
    folder = Path(folder_path)
    all_files = []
    
    for ext in extensions:
        all_files.extend(folder.glob(f'*{ext}'))
        # all_files.extend(folder.glob(f'*{ext.upper()}'))
    
    return sorted([str(f) for f in all_files])

# Pakai:
DOCUMENTS_FOLDER = r'E:\Softwares\Jupyter\Projects\OCR\data\dokumen\listrik'
FILE_PATHS = get_all_documents(DOCUMENTS_FOLDER)
print(f"✅ Ditemukan {len(FILE_PATHS)} dokumen")

# Tampilkan daftar file (dibatasi MAX_DISPLAY_INPUT)
print(f"\n📋 Daftar file input:")
for i, fp in enumerate(FILE_PATHS):
    if MAX_DISPLAY_INPUT is not None and i >= MAX_DISPLAY_INPUT:
        print(f"   ... dan {len(FILE_PATHS) - MAX_DISPLAY_INPUT} file lainnya")
        break
    print(f"   [{i+1}] {os.path.basename(fp)}")

In [ ]:
# ================================================
# KONVERSI FILE KE GAMBAR
# ================================================

print("🔄 Memproses file...")
print("=" * 60)

# Inisialisasi timer global & tracking per dokumen
ocr_pipeline_start = time.time()
timing_per_doc = {}

def _record_timing(doc_name, step, duration):
    """Catat waktu proses per dokumen per tahap"""
    if doc_name not in timing_per_doc:
        timing_per_doc[doc_name] = {}
    timing_per_doc[doc_name][step] = timing_per_doc[doc_name].get(step, 0) + duration

all_images = []
file_info = []  # Track source file untuk setiap gambar

for idx_file, file_path in enumerate(FILE_PATHS):
    print(f"\n📄 Memproses: {file_path}")
    _t_doc = time.time()
    
    try:
        if file_path.lower().endswith('.pdf'):
            # Konversi PDF ke gambar
            images = convert_from_path(
                file_path,
                dpi=300,
                poppler_path=POPPLER_PATH
            )
            print(f"   ✅ PDF dikonversi ke {len(images)} halaman")
        else:
            # Baca file gambar langsung
            images = [Image.open(file_path)]
            print(f"   ✅ Gambar berhasil dibaca")
        
        # Simpan semua gambar dan info file
        for img in images:
            all_images.append(img)
            file_info.append(os.path.basename(file_path))
        
        _record_timing(os.path.basename(file_path), 'Konversi', time.time() - _t_doc)
        
        # Preview halaman pertama (dibatasi MAX_DISPLAY_INPUT)
        if images:
            if MAX_DISPLAY_INPUT is None or idx_file < MAX_DISPLAY_INPUT:
                print(f"\n   🔍 Preview (Halaman 1):")
                display(images[0])
            else:
                if idx_file == MAX_DISPLAY_INPUT:
                    print(f"\n   💡 Preview tidak ditampilkan untuk sisa {len(FILE_PATHS) - MAX_DISPLAY_INPUT} file")
            
    except Exception as e:
        print(f"   ❌ Error: {e}")
        continue

print(f"\n{'=' * 60}")
print(f"📊 Total gambar siap diproses: {len(all_images)}")
print(f"{'=' * 60}")

images = all_images

In [ ]:
# ================================================
# PREPROCESSING 0: ORIENTATION & SKEW CORRECTION
# ================================================
import matplotlib.pyplot as plt

MAX_DISPLAY_ORIENT = 5

print("🔄 Memulai proses Orientation & Skew Correction...")
print("=" * 60)
start_time = time.time()

corrected_images = []

for i, img in enumerate(images):
    _t_doc = time.time()
    source_file = file_info[i]
    rotation_applied = False
    skew_applied = False
    rotation_angle = 0
    rotation_conf = 0.0
    skew_angle = 0.0
    osd_error = None

    # === STEP 1: Deteksi & Koreksi Rotasi (OSD) ===
    try:
        osd = pytesseract.image_to_osd(img)
        for line in osd.split('\n'):
            if 'Rotate:' in line:
                rotation_angle = int(line.split(':')[-1].strip())
            if 'Orientation confidence:' in line:
                rotation_conf = float(line.split(':')[-1].strip())
    except pytesseract.TesseractError as e:
        osd_error = str(e)

    # Rotasi jika terdeteksi (confidence > 1.0)
    if osd_error is None and rotation_angle != 0 and rotation_conf > 1.0:
        img = img.rotate(rotation_angle, expand=True, fillcolor=(255, 255, 255))
        rotation_applied = True

    # === STEP 1b: Fallback Rotasi 180° ===
    # Jika OSD gagal atau tidak mendeteksi rotasi, cek apakah dokumen terbalik
    # dengan membandingkan OCR confidence normal vs rotasi 180°
    if not rotation_applied:
        try:
            img_np_temp = np.array(img)
            h, w = img_np_temp.shape[:2]
            # Ambil crop tengah untuk tes cepat
            center_crop = img_np_temp[h//4:3*h//4, w//4:3*w//4]
            
            # Confidence orientasi normal
            data_normal = pytesseract.image_to_data(
                Image.fromarray(center_crop),
                lang='ind+eng',
                config='--psm 6 --oem 3',
                output_type=pytesseract.Output.DICT
            )
            conf_normal = [int(c) for c in data_normal['conf'] if int(c) > 0]
            avg_conf_normal = sum(conf_normal) / len(conf_normal) if conf_normal else 0
            
            # Confidence rotasi 180°
            rotated_180 = np.rot90(img_np_temp, 2)
            center_crop_180 = rotated_180[h//4:3*h//4, w//4:3*w//4]
            data_rotated = pytesseract.image_to_data(
                Image.fromarray(center_crop_180),
                lang='ind+eng',
                config='--psm 6 --oem 3',
                output_type=pytesseract.Output.DICT
            )
            conf_rotated = [int(c) for c in data_rotated['conf'] if int(c) > 0]
            avg_conf_rotated = sum(conf_rotated) / len(conf_rotated) if conf_rotated else 0
            
            # Rotasi 180° jika confidence jauh lebih tinggi
            if avg_conf_rotated > avg_conf_normal + 10:
                img = Image.fromarray(rotated_180)
                rotation_applied = True
                rotation_angle = 180
                rotation_conf = avg_conf_rotated
        except:
            pass  # Fallback gagal, lanjut dengan orientasi asli

    # === STEP 2: Deteksi & Koreksi Skew (Kemiringan Kecil) ===
    img_np = np.array(img)
    gray_temp = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY) if len(img_np.shape) == 3 else img_np

    _, binary = cv2.threshold(gray_temp, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    coords = np.column_stack(np.where(binary > 0))

    if len(coords) >= 10:
        angle = cv2.minAreaRect(coords)[-1]
        skew_angle = -(90 + angle) if angle < -45 else -angle
        if abs(skew_angle) > 15:
            skew_angle = 0.0

    if abs(skew_angle) >= 0.5:
        (h, w) = img_np.shape[:2]
        M = cv2.getRotationMatrix2D((w // 2, h // 2), skew_angle, 1.0)
        border = (255, 255, 255) if len(img_np.shape) == 3 else 255
        img_np = cv2.warpAffine(img_np, M, (w, h), flags=cv2.INTER_CUBIC,
                                borderMode=cv2.BORDER_CONSTANT, borderValue=border)
        skew_applied = True

    corrected_pil = Image.fromarray(img_np)
    corrected_images.append(corrected_pil)
    _record_timing(source_file, 'Orientasi', time.time() - _t_doc)

    # === DISPLAY ===
    should_display = (MAX_DISPLAY_ORIENT is None) or (i < MAX_DISPLAY_ORIENT)

    if should_display:
        print(f"\n✅ Orientation & Skew Correction gambar {i+1} dari {len(images)}:")
        print(f"   📄 File: {source_file}")

        if osd_error:
            print(f"   ⚠️  OSD Error: {osd_error}")
        
        if rotation_angle == 180 and rotation_applied:
            print(f"   🔄 Rotasi 180° terdeteksi via confidence check → Dikoreksi")
        elif rotation_applied:
            print(f"   🔍 Rotation: {rotation_angle}° (conf: {rotation_conf:.2f}) → Dikoreksi")
        else:
            print(f"   🔍 Rotation: Tidak perlu")

        print(f"   🔍 Skew: {skew_angle:.2f}° → {'Dikoreksi' if skew_applied else 'Tidak perlu'}")

        fig, axes = plt.subplots(1, 2, figsize=(12, 6))
        axes[0].imshow(np.array(images[i]))
        axes[0].set_title('Original')
        axes[0].axis('off')
        axes[1].imshow(np.array(corrected_pil))
        axes[1].set_title(f'Corrected (rot:{rotation_angle}° skew:{skew_angle:.1f}°)')
        axes[1].axis('off')
        plt.tight_layout()
        plt.show()
        print("-" * 60)
    else:
        if i == MAX_DISPLAY_ORIENT:
            print(f"\n📊 Memproses gambar {i+1} - {len(images)}...")
        print(f"   ✅ Gambar {i+1} selesai", end="\r")

if MAX_DISPLAY_ORIENT is not None and len(images) > MAX_DISPLAY_ORIENT:
    print(f"\n\n💡 {len(images) - MAX_DISPLAY_ORIENT} gambar lainnya sudah diproses")

elapsed = time.time() - start_time
print(f"\n⏱️  Waktu Orientation & Skew Correction: {elapsed:.2f} detik")
print(f"📊 Total gambar diproses: {len(corrected_images)}")
print("=" * 60)

In [ ]:
# ================================================
# PREPROCESSING 1: GRAYSCALING
# ================================================

MAX_DISPLAY_GRAY = 5

print("🔄 Memulai proses Grayscaling...")
print("=" * 60)
start_time = time.time()

grayscale_images = []

for i, img in enumerate(corrected_images):
    _t_doc = time.time()
    open_cv_image = np.array(img)
    
    # Konversi RGB ke Grayscale
    if len(open_cv_image.shape) == 3:
        img_gray = cv2.cvtColor(open_cv_image, cv2.COLOR_RGB2GRAY)
    else:
        img_gray = open_cv_image
    
    grayscale_images.append(img_gray)
    _record_timing(file_info[i], 'Grayscale', time.time() - _t_doc)

    should_display = (MAX_DISPLAY_GRAY is None) or (i < MAX_DISPLAY_GRAY)

    if should_display:
        print(f"\n✅ Grayscale gambar {i+1} dari {len(corrected_images)}:")
        print(f"   Dimensi: {img_gray.shape[1]} x {img_gray.shape[0]} pixels")
        display(Image.fromarray(img_gray))
        print("-" * 60)
    else:
        if i == MAX_DISPLAY_GRAY:
            print(f"\n📊 Memproses gambar {i+1} - {len(corrected_images)}...")
        print(f"   ✅ Gambar {i+1} selesai", end="\r")

if MAX_DISPLAY_GRAY is not None and len(corrected_images) > MAX_DISPLAY_GRAY:
    print(f"\n\n💡 {len(corrected_images) - MAX_DISPLAY_GRAY} gambar lainnya sudah diproses")

elapsed = time.time() - start_time
print(f"\n⏱️  Waktu Grayscaling: {elapsed:.2f} detik")
print(f"📊 Total gambar diproses: {len(grayscale_images)}")
print("=" * 60)

In [ ]:
# ================================================
# PREPROCESSING 2: ADAPTIVE NOISE REMOVAL
# ================================================
import matplotlib.pyplot as plt
MAX_DISPLAY_DENOISE = 5
print('Memulai proses Adaptive Noise Removal...')
print('=' * 60)
start_time = time.time()

denoised_images = []

for i, gray_img in enumerate(grayscale_images):
    _t_doc = time.time()
    is_jpg = file_info[i].lower().endswith(('.jpg', '.jpeg'))

    if is_jpg:
        # Foto fisik: Non-local means mempertahankan tepi teks
        denoised = cv2.fastNlMeansDenoising(gray_img, h=10)
        method = 'NlMeans (JPG)'
    else:
        # PDF digital: cek noise level
        noise_level = cv2.Laplacian(gray_img, cv2.CV_64F).var()
        if noise_level > 1500:
            denoised = cv2.GaussianBlur(gray_img, (3, 3), 0)
            method = 'Gaussian 3x3 (noisy PDF)'
        else:
            denoised = gray_img.copy()
            method = 'No blur (clean)'

    denoised_images.append(denoised)
    _record_timing(file_info[i], 'Denoise', time.time() - _t_doc)

    should_display = (MAX_DISPLAY_DENOISE is None) or (i < MAX_DISPLAY_DENOISE)
    if should_display:
        print(f'\nDenoise gambar {i+1} dari {len(grayscale_images)}:')
        print(f'   Dimensi: {denoised.shape[1]} x {denoised.shape[0]} pixels')
        print(f'   Metode: {method}')
        fig, axes = plt.subplots(1, 2, figsize=(12, 6))
        axes[0].imshow(gray_img, cmap='gray')
        axes[0].set_title('Grayscale')
        axes[0].axis('off')
        axes[1].imshow(denoised, cmap='gray')
        axes[1].set_title(f'Denoised ({method})')
        axes[1].axis('off')
        plt.tight_layout()
        plt.show()
        print('-' * 60)
    else:
        if i == MAX_DISPLAY_DENOISE:
            print(f'\nMemproses gambar {i+1} - {len(grayscale_images)}...')
        print(f'   Gambar {i+1} selesai', end='\r')

elapsed = time.time() - start_time
print(f'\nWaktu Adaptive Noise Removal: {elapsed:.2f} detik')
print(f'Total gambar diproses: {len(denoised_images)}')
print('=' * 60)


In [ ]:
# ================================================
# PREPROCESSING 5: THRESHOLDING
# Adaptive untuk JPG, Pass-through untuk PDF
# ================================================
print('Memulai proses Thresholding...')
print('=' * 60)
start_time = time.time()

thresh_images_adaptive = []

for i, img in enumerate(denoised_images):
    _t_doc = time.time()
    is_jpg = file_info[i].lower().endswith(('.jpg', '.jpeg'))

    if is_jpg:
        # Foto fisik: adaptive threshold lokal untuk pencahayaan tidak merata
        final = cv2.adaptiveThreshold(
            img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 10
        )
        method = 'Adaptive Gaussian (JPG)'
    else:
        # PDF: render digital bersih — Tesseract LSTM lebih akurat pada grayscale
        final = img
        method = 'Pass-through (PDF)'

    thresh_images_adaptive.append(final)
    _record_timing(file_info[i], 'Threshold', time.time() - _t_doc)

elapsed = time.time() - start_time
print(f'Total gambar: {len(thresh_images_adaptive)} ({elapsed:.2f} detik)')
print('=' * 60)


In [ ]:
# ================================================
# EKSTRAKSI TEKS DENGAN TESSERACT + POST-PROCESSING
# ================================================
print('Memulai ekstraksi teks dengan Tesseract...')
print('   Config: PSM 4 (single column) + OEM 3 (LSTM), fallback PSM 3')
print('=' * 60)
start_time_total = time.time()

extracted_texts = {}

def clean_garbage_lines(text):
    lines = text.split('\n')
    cleaned_lines = []
    for line in lines:
        stripped = line.strip()
        if not stripped:
            cleaned_lines.append(line)
            continue
        is_garbage = False
        # Cek 1: Baris sangat pendek
        if len(stripped) <= 2 and not stripped.isdigit():
            is_garbage = True
        # Cek 2: Rasio alfanumerik rendah
        if not is_garbage:
            alnum = sum(1 for c in stripped if c.isalnum())
            if alnum / len(stripped) < 0.4 and len(stripped) < 20:
                is_garbage = True
        # Cek 3: Banyak karakter tunggal
        if not is_garbage:
            words = stripped.split()
            if len(words) >= 3:
                single_ratio = sum(1 for w in words if len(w) == 1) / len(words)
                if single_ratio > 0.6 and len(stripped) < 30:
                    is_garbage = True
        # Cek 4: Hanya simbol tabel
        if not is_garbage:
            non_sym = stripped.replace('|', '').replace('-', '').replace('=', '').strip()
            if len(non_sym) < 3 and len(stripped) > 2:
                is_garbage = True
        if not is_garbage:
            cleaned_lines.append(line)
    return '\n'.join(cleaned_lines)


def extract_text(img, lang='ind+eng'):
    # PSM 4 = single column — lebih stabil untuk struk tagihan
    text = pytesseract.image_to_string(img, lang=lang, config='--psm 4 --oem 3')
    # Fallback ke PSM 3 jika output terlalu sedikit
    if sum(c.isalnum() for c in text) < 100:
        text_alt = pytesseract.image_to_string(img, lang=lang, config='--psm 3 --oem 3')
        if sum(c.isalnum() for c in text_alt) > sum(c.isalnum() for c in text):
            text = text_alt
    return text


for i, final_img in enumerate(thresh_images_adaptive):
    start_time = time.time()

    text = extract_text(final_img)
    text = re.sub(r' +', ' ', text)
    text = re.sub(r' +\n', '\n', text)
    text = clean_garbage_lines(text)
    text = re.sub(r'\n\s*\n+', '\n\n', text)
    text = text.strip()

    source_file = file_info[i]
    if source_file not in extracted_texts:
        extracted_texts[source_file] = []
    extracted_texts[source_file].append(text)

    elapsed = time.time() - start_time
    _record_timing(source_file, 'Ekstraksi', elapsed)

    print(f'\nHalaman {i+1}/{len(thresh_images_adaptive)} (File: {source_file})')
    print(f'   Waktu: {elapsed:.2f} detik | Karakter: {len(text)}')
    print(f'   Preview: {text[:200]}...')
    print('-' * 60)

elapsed_total = time.time() - start_time_total
avg_time = elapsed_total / len(thresh_images_adaptive)
print(f'\nTotal waktu ekstraksi: {elapsed_total:.2f} detik')
print(f'Rata-rata: {avg_time:.2f} detik/halaman')
if avg_time < 10:
    print('Target performa tercapai (<10 detik/dokumen)')
else:
    print('Performa belum optimal (target: <10 detik)')
print('Output disimpan di variable: extracted_texts')
print('=' * 60)


In [ ]:
# ================================================
# OUTPUT LIST UNTUK MODUL NER
# ================================================
# Format output: List of dict, setiap item berisi:
#   - 'nama_lampiran': Nama file dokumen (string)
#   - 'hasil_ocr': Teks hasil OCR (string)
#
# List ini bisa langsung diteruskan ke modul NER
# untuk ekstraksi entitas (nominal, IDPEL, nama, dll)

ocr_results_list = []

for filename, ocr_texts in extracted_texts.items():
    # Gabungkan semua halaman jadi satu teks
    full_text = '\n'.join(ocr_texts)
    
    ocr_results_list.append({
        'nama_lampiran': filename,
        'hasil_ocr': full_text
    })

# Tampilkan hasil
print("=" * 60)
print("📋 OUTPUT LIST UNTUK MODUL NER")
print("=" * 60)
print(f"\n📊 Total dokumen: {len(ocr_results_list)}")
print(f"📦 Variable: ocr_results_list\n")

for i, item in enumerate(ocr_results_list, 1):
    preview = item['hasil_ocr'][:100].replace('\n', ' ')
    print(f"  [{i}] {item['nama_lampiran']}")
    print(f"      Preview: {preview}...")
    print(f"      Panjang: {len(item['hasil_ocr'])} karakter")
    print()

print("=" * 60)
print("💡 Gunakan ocr_results_list untuk input ke modul NER")
print("   Contoh akses:")
print("   >>> ocr_results_list[0]['nama_lampiran']")
print("   >>> ocr_results_list[0]['hasil_ocr']")
print("=" * 60)

In [ ]:
# ============================================
# RINGKASAN WAKTU KESELURUHAN OCR PIPELINE
# ============================================
ocr_pipeline_total = time.time() - ocr_pipeline_start

print('=' * 100)
print('RINGKASAN WAKTU OCR PIPELINE')
print('=' * 100)

STEPS = ['Konversi', 'Orientasi', 'Grayscale', 'Denoise', 'Threshold', 'Ekstraksi']

header = f"{'No':<4} {'Dokumen':<35} "
for step in STEPS:
    header += f"{step:<12} "
header += f"{'TOTAL':<10}"
print(f'\n{header}')
print('-' * 100)

doc_totals = []
for idx, (doc_name, steps) in enumerate(timing_per_doc.items(), 1):
    row = f"{idx:<4} {doc_name:<35} "
    total = 0
    for step in STEPS:
        t = steps.get(step, 0)
        total += t
        row += f"{t:<12.2f} "
    row += f"{total:<10.2f}"
    doc_totals.append(total)
    print(row)

print('-' * 100)
avg_row = f"{'RATA-RATA':<39} "
for step in STEPS:
    avg_val = sum(timing_per_doc[d].get(step, 0) for d in timing_per_doc) / len(timing_per_doc)
    avg_row += f"{avg_val:<12.2f} "
avg_total = sum(doc_totals) / len(doc_totals) if doc_totals else 0
avg_row += f"{avg_total:<10.2f}"
print(avg_row)
print('=' * 100)

print(f'\nTotal waktu keseluruhan OCR pipeline: {ocr_pipeline_total:.2f} detik')
print(f'Rata-rata per dokumen: {avg_total:.2f} detik')
if avg_total < 10:
    print('Target performa tercapai (<10 detik/dokumen)')
else:
    print('Performa belum optimal (target: <10 detik/dokumen)')
print('Satuan waktu: detik (seconds)')
print('=' * 100)


## Pengujian Akurasi, WER, dan CER menggunakan jiwer

In [ ]:
# ================================================
# FUNGSI UNTUK MENGHITUNG METRIK (SINGLE-LINE)
# ================================================
# WER (Word Error Rate): error level kata
# CER (Character Error Rate): error level karakter
# Nilai lebih rendah = lebih baik. 0% = sempurna.

def normalize_to_single_line(text):
    return re.sub(r'\s+', ' ', text.strip())

def calculate_wer_cer_jiwer(ground_truth, ocr_output):
    gt_line  = normalize_to_single_line(ground_truth)
    ocr_line = normalize_to_single_line(ocr_output)
    if not gt_line or not ocr_line:
        return {'wer': 100.0, 'cer': 100.0}
    return {
        'wer': wer(gt_line, ocr_line) * 100,
        'cer': cer(gt_line, ocr_line) * 100
    }

print('Fungsi metrik pengujian siap digunakan (mode: SINGLE-LINE)')
print('WER: jiwer library (error level kata)')
print('CER: jiwer library (error level karakter)')


In [ ]:
# ================================================
# PENGUJIAN WER DAN CER (SINGLE-LINE)
# ================================================
print('\n' + '=' * 80)
print('PENGUJIAN AKURASI OCR (Mode: Single-Line)')
print('=' * 80)

testing_results = []

for filename, ocr_texts in extracted_texts.items():
    print(f'\n{"=" * 80}')
    print(f'Testing File: {filename}')
    print(f'{"=" * 80}')

    if filename not in GROUND_TRUTH:
        print(f'Ground truth tidak tersedia untuk {filename}')
        continue

    ground_truth = GROUND_TRUTH[filename]
    ocr_output   = '\n'.join(ocr_texts)

    gt_single  = normalize_to_single_line(ground_truth)
    ocr_single = normalize_to_single_line(ocr_output)

    print(f'\nInformasi Dasar (Single-Line):')
    print(f'   Ground Truth: {len(gt_single)} karakter')
    print(f'   OCR Output:   {len(ocr_single)} karakter')
    print(f'   Selisih:      {abs(len(gt_single) - len(ocr_single))} karakter')

    metrics = calculate_wer_cer_jiwer(ground_truth, ocr_output)
    print(f'\nWER: {metrics["wer"]:.2f}%')
    print(f'CER: {metrics["cer"]:.2f}%')

    testing_results.append({
        'filename': filename,
        'ground_truth_length': len(gt_single),
        'ocr_output_length':   len(ocr_single),
        'wer': metrics['wer'],
        'cer': metrics['cer'],
        'ground_truth': ground_truth,
        'ocr_output':   ocr_output
    })

    print(f'\nPreview Single-Line (150 karakter pertama):')
    print(f'   GT:  {gt_single[:150]}...')
    print(f'   OCR: {ocr_single[:150]}...')

print(f'\n\n{"=" * 80}')
print('PENGUJIAN SELESAI')
print(f'{"=" * 80}')


In [ ]:
# ================================================
# RINGKASAN HASIL PENGUJIAN (SINGLE-LINE)
# ================================================
if testing_results:
    print('\n' + '=' * 70)
    print('RINGKASAN HASIL PENGUJIAN SEMUA DOKUMEN (Single-Line)')
    print('=' * 70)

    avg_wer = sum(r['wer'] for r in testing_results) / len(testing_results)
    avg_cer = sum(r['cer'] for r in testing_results) / len(testing_results)

    print(f"\n{'No':<4} {'Dokumen':<40} {'WER (%)':<12} {'CER (%)':<12}")
    print('-' * 70)
    for i, result in enumerate(testing_results, 1):
        print(f"{i:<4} {result['filename']:<40} "
              f"{result['wer']:<12.2f} "
              f"{result['cer']:<12.2f}")
    print('-' * 70)
    print(f"{'RATA-RATA':<44} {avg_wer:<12.2f} {avg_cer:<12.2f}")
    print('=' * 70)

    print('\nInterpretasi Hasil:')
    if avg_wer <= 10:
        print(f'   WER: SANGAT BAIK ({avg_wer:.2f}%)')
    elif avg_wer <= 20:
        print(f'   WER: BAIK ({avg_wer:.2f}%)')
    elif avg_wer <= 30:
        print(f'   WER: CUKUP ({avg_wer:.2f}%)')
    else:
        print(f'   WER: KURANG ({avg_wer:.2f}%)')

    if avg_cer <= 5:
        print(f'   CER: SANGAT BAIK ({avg_cer:.2f}%)')
    elif avg_cer <= 10:
        print(f'   CER: BAIK ({avg_cer:.2f}%)')
    elif avg_cer <= 15:
        print(f'   CER: CUKUP ({avg_cer:.2f}%)')
    else:
        print(f'   CER: KURANG ({avg_cer:.2f}%)')

    print('\nCatatan:')
    print('   Mode evaluasi: SINGLE-LINE (layout/newline diabaikan)')
    print('   WER: Kesalahan level kata (semakin rendah semakin baik)')
    print('   CER: Kesalahan level karakter (semakin rendah semakin baik)')
else:
    print('Tidak ada hasil pengujian. Pastikan ground truth sudah dikonfigurasi.')


In [ ]:
# ================================================
# ANALISIS DETAIL PER DOKUMEN
# ================================================
if testing_results:
    print('=' * 80)
    print('ANALISIS DETAIL PER DOKUMEN')
    print('=' * 80)

    for i, result in enumerate(testing_results, 1):
        sep = chr(8212) * 80
        print(f'\n{sep}')
        print(f'[{i}] {result["filename"]}')
        print(f'   WER: {result["wer"]:.2f}% | CER: {result["cer"]:.2f}%')
        print(f'   GT: {result["ground_truth_length"]} chars | '
              f'OCR: {result["ocr_output_length"]} chars | '
              f'Selisih: {result["ocr_output_length"] - result["ground_truth_length"]:+d} chars')
        print(f'{sep}')

        gt_single  = normalize_to_single_line(result['ground_truth'])
        ocr_single = normalize_to_single_line(result['ocr_output'])

        max_display = 500
        print('\n   Ground Truth (single-line):')
        if len(gt_single) > max_display:
            print(f'   {gt_single[:max_display]}... [{len(gt_single)} total chars]')
        else:
            print(f'   {gt_single}')

        print('\n   OCR Output (single-line):')
        if len(ocr_single) > max_display:
            print(f'   {ocr_single[:max_display]}... [{len(ocr_single)} total chars]')
        else:
            print(f'   {ocr_single}')

        gt_words  = set(gt_single.lower().split())
        ocr_words = set(ocr_single.lower().split())
        common  = gt_words & ocr_words
        missing = gt_words - ocr_words
        extra   = ocr_words - gt_words

        print('\n   Analisis Kata:')
        if gt_words:
            print(f'      Kata cocok:    {len(common)}/{len(gt_words)} ({len(common)/len(gt_words)*100:.0f}%)')
        if missing:
            print(f'      Kata hilang:   {len(missing)} — contoh: {", ".join(list(missing)[:10])}')
        if extra:
            print(f'      Kata tambahan: {len(extra)} — contoh: {", ".join(list(extra)[:10])}')

    print(f'\n\n{"=" * 80}')
    print('KESIMPULAN ANALISIS')
    print(f'{"=" * 80}')

    best     = min(testing_results, key=lambda r: r['cer'])
    worst    = max(testing_results, key=lambda r: r['cer'])
    avg_cer_val = sum(r['cer'] for r in testing_results) / len(testing_results)

    print(f'\n   Terbaik  (CER terendah): {best["filename"]} ({best["cer"]:.2f}%)')
    print(f'   Terburuk (CER tertinggi): {worst["filename"]} ({worst["cer"]:.2f}%)')
    print(f'\n   Rata-rata CER: {avg_cer_val:.2f}%')

    if avg_cer_val <= 10:
        print('\n   CER di bawah 10% — cukup baik untuk verifikasi nominal')
    elif avg_cer_val <= 20:
        print('\n   CER 10-20% — perlu perbaikan untuk verifikasi yang akurat')
    else:
        print('\n   CER di atas 20% — masih perlu optimasi lebih lanjut')

    print('\n   Tips peningkatan:')
    print('      Kualitas scan/foto sangat mempengaruhi hasil OCR')
    print('      Dokumen dengan watermark/stempel cenderung memiliki CER lebih tinggi')
    print('      Dokumen digital bersih bisa mencapai CER <1%')
else:
    print('Tidak ada hasil testing untuk dianalisis.')
